# Tony: DQN Trading Bot - Analysis

Interactive analysis of training results and agent performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import yaml
from stable_baselines3 import DQN
from env.trading_env import TradingEnv

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

## Load Config & Data

In [ ]:
with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

test_data = np.load('../data/test.npz', allow_pickle=True)
print('Test data keys:', list(test_data.keys()))
print('Test data length:', len(test_data['close_prices']))

## Run Agent on Test Set

In [ ]:
env = TradingEnv(
    close_prices=test_data['close_prices'],
    pct_changes=test_data['pct_changes'],
    sma_ratios=test_data['sma_ratios'],
    rsi_norm=test_data['rsi_norm'],
    window_size=config['env']['window_size'],
    episode_length=config['env']['episode_length'],
    initial_cash=config['env']['initial_cash'],
    transaction_cost=config['env']['transaction_cost'],
    max_drawdown_threshold=config['env']['max_drawdown_threshold'],
    mode='test',
)

model = DQN.load('../models/best_model')
obs, _ = env.reset()
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = env.step(int(action))
    done = terminated or truncated

metrics = env.get_episode_metrics()
for k, v in metrics.items():
    print(f'{k}: {v}')

## Action Distribution

In [ ]:
actions = np.array(env.actions_taken)
labels = ['Buy', 'Hold', 'Sell']
counts = [np.sum(actions == i) for i in range(3)]

plt.bar(labels, counts, color=['green', 'gray', 'red'])
plt.title('Action Distribution')
plt.ylabel('Count')
for i, c in enumerate(counts):
    plt.text(i, c + 1, str(c), ha='center')
plt.show()

print(f'Total actions: {len(actions)}')
for label, count in zip(labels, counts):
    print(f'  {label}: {count} ({count/len(actions):.1%})')

## Portfolio Value Over Time

In [ ]:
plt.plot(env.portfolio_values)
plt.xlabel('Step')
plt.ylabel('Portfolio Value ($)')
plt.title('DQN Agent Portfolio Value')
plt.grid(True, alpha=0.3)
plt.show()

## TensorBoard

Launch TensorBoard to view training curves:

```bash
tensorboard --logdir ../runs/
```

Then open http://localhost:6006 in your browser.

## Phase 2 Ideas

- Multi-asset support
- PPO / A2C comparison
- Continuous action space (position sizing)
- More technical indicators (MACD, Bollinger Bands)
- Walk-forward validation
- Live paper trading integration